# DOS-RAG vs Vanilla RAG: Subset Audit and Next Directions

This notebook summarizes three things:

1. what this repo actually implemented for `vanilla_rag` and `dos_rag`
2. what happened on the saved subset benchmark in `outputs_meeting_core/subset`
3. what is still unclear, what to test next, and what papers are most worth reading now

This notebook is based on the current repo state, especially:

- `benchmarking/rag_pipeline.py`
- `benchmarking/official_methods.py`
- `outputs_meeting_core/subset/*`
- `docs/LAST_WEEK_RESULTS.md`

Important implementation nuance:

- `vanilla_rag` is executed through the repo-owned retrieval pipeline.
- `dos_rag` in the saved benchmark results is executed through the official DOS adapter in `OfficialMethodRunner`, not through the generic retrieval helper.
- Both are still scored by the same official SCROLLS metric path and use the same shared local reader model in this benchmark harness.


In [3]:
from pathlib import Path
import json
import sys
from statistics import mean

try:
    from IPython.display import display
except ModuleNotFoundError:
    def display(obj):
        print(obj)

try:
    import pandas as pd
except ModuleNotFoundError:
    pd = None


def find_repo_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for candidate in [start, *start.parents]:
        if (candidate / "benchmarking").exists() and (candidate / "outputs_meeting_core").exists():
            return candidate
    raise FileNotFoundError("Could not locate the repo root from the current working directory.")


ROOT = find_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

try:
    from benchmarking.metrics import compute_metrics
except ModuleNotFoundError as exc:
    raise RuntimeError(
        "Activate the repo environment first, e.g. `source venv/bin/activate`, before running this notebook."
    ) from exc

RUN_ROOT = ROOT / "outputs_meeting_core" / "subset"
TASKS = ["qmsum", "qasper", "narrative_qa", "quality", "contract_nli"]
METHODS = ["vanilla_rag", "dos_rag"]


def load_json(path):
    with open(path) as fh:
        return json.load(fh)


def load_jsonl(path):
    rows = []
    with open(path) as fh:
        for line in fh:
            if line.strip():
                rows.append(json.loads(line))
    return rows


def as_table(rows):
    if pd is not None:
        return pd.DataFrame(rows)
    return rows


def score_row(task, row):
    return compute_metrics(
        [row.get("prediction", "")],
        [row.get("references", [])],
        task_name=task,
    ).get("scrolls_score", 0.0)


print("Repo root:", ROOT)
print("Run root:", RUN_ROOT)


Repo root: /home/yifan/FORWARD Data Lab/long-document-qa-baseline
Run root: /home/yifan/FORWARD Data Lab/long-document-qa-baseline/outputs_meeting_core/subset


## What We Implemented

### `vanilla_rag`

The repo baseline path is:

1. chunk the document with the repo chunker
2. embed chunks with the shared dense embedder
3. retrieve top chunks with FAISS by semantic similarity
4. keep adding retrieved chunks until the shared `context_budget` is hit
5. present the context in **retrieval-rank order**
6. answer with the shared local reader

This is implemented in the repo-owned benchmark path in `benchmarking/rag_pipeline.py`.

### `dos_rag`

The saved DOS results are produced by the official DOS adapter path in `benchmarking/official_methods.py`:

1. build the DOS index using the official `dos-rag-eval` retrieval core
2. retrieve chunks under the same shared `top_k` and `context_budget`
3. restore the selected chunks to **document order**
4. answer with the same shared local reader used by the benchmark

### Controlled comparison point

The important scientific comparison here is:

- same benchmark
- same reader
- same scoring
- same token budget
- different retrieval path and, crucially, different **prompt ordering**

For the saved subset results, the observed prompt ordering is consistent across all tasks:

- `vanilla_rag`: `retrieval_rank`
- `dos_rag`: `document_order`

That means the current benchmark is not just testing retrieval quality. It is also testing what happens when the same or nearly the same evidence is arranged in a more source-faithful order.


In [5]:
score_rows = []
for task in TASKS:
    row = {"task": task}
    for method in METHODS:
        summary = load_json(RUN_ROOT / method / task / "summary.json")
        row[f"{method}_score"] = round(summary["metrics"]["scrolls_score"], 4)
        row[f"{method}_avg_input_tokens"] = round(summary["token_stats"]["avg_input_tokens"], 2)
        row[f"{method}_avg_context_tokens"] = round(summary["token_stats"]["avg_context_tokens"], 2)
        row[f"{method}_elapsed_seconds"] = round(summary["elapsed_seconds"], 2)
    row["dos_minus_vanilla"] = round(row["dos_rag_score"] - row["vanilla_rag_score"], 4)
    score_rows.append(row)

overall = {
    "task": "average",
    "vanilla_rag_score": round(mean(r["vanilla_rag_score"] for r in score_rows), 4),
    "dos_rag_score": round(mean(r["dos_rag_score"] for r in score_rows), 4),
}
overall["dos_minus_vanilla"] = round(overall["dos_rag_score"] - overall["vanilla_rag_score"], 4)

display(as_table(score_rows + [overall]))


,task,vanilla_rag_score,vanilla_rag_avg_input_tokens,vanilla_rag_avg_context_tokens,vanilla_rag_elapsed_seconds,dos_rag_score,dos_rag_avg_input_tokens,dos_rag_avg_context_tokens,dos_rag_elapsed_seconds,dos_minus_vanilla
0,qmsum,16.6505,9105.50,9049.06,249.63,18.2178,9159.66,9103.36,565.36,1.5673
1,qasper,51.4486,4987.66,4858.78,86.65,54.4233,5029.38,4901.36,95.39,2.9747
2,narrative_qa,19.5300,10084.12,9953.22,461.07,21.6398,9713.14,9585.54,1125.02,2.1098
3,quality,70.0000,6484.34,6310.76,109.76,78.0000,6599.58,6428.58,87.73,8.0000
4,contract_nli,72.0000,3950.00,3842.04,65.14,66.0000,3977.50,3870.84,36.33,-6.0000
5,average,45.9258,NaN,NaN,NaN,47.6562,NaN,NaN,NaN,1.7304


## Subset Result Summary

Meeting-ready summary from `outputs_meeting_core/subset`:

- overall, DOS beats vanilla on the subset average: `47.6562` vs `45.9258` (`+1.7304`)
- strongest gain is on `quality`: `+8.0`
- DOS also improves `qmsum`, `qasper`, and `narrative_qa`
- DOS regresses on `contract_nli` by `-6.0`

So the effect is clearly **task-dependent**, not uniformly positive.


In [6]:
phen_rows = []

for task in TASKS:
    vanilla_rows = {row["id"]: row for row in load_jsonl(RUN_ROOT / "vanilla_rag" / task / "results.jsonl")}
    dos_rows = {row["id"]: row for row in load_jsonl(RUN_ROOT / "dos_rag" / task / "results.jsonl")}
    shared_ids = sorted(set(vanilla_rows) & set(dos_rows))

    dos_better = 0
    vanilla_better = 0
    tied = 0
    near_full_dos = 0
    dos_contiguous = 0
    selected_jaccards = []

    for ex_id in shared_ids:
        vanilla = vanilla_rows[ex_id]
        dos = dos_rows[ex_id]

        vanilla_score = score_row(task, vanilla)
        dos_score = score_row(task, dos)
        if dos_score > vanilla_score:
            dos_better += 1
        elif vanilla_score > dos_score:
            vanilla_better += 1
        else:
            tied += 1

        if dos.get("document_tokens") and (dos.get("context_tokens", 0) / dos["document_tokens"]) >= 0.95:
            near_full_dos += 1

        indices = dos.get("selected_chunk_indices", [])
        if indices and indices == list(range(min(indices), max(indices) + 1)):
            dos_contiguous += 1

        vanilla_set = set(vanilla.get("selected_chunk_indices", []))
        dos_set = set(dos.get("selected_chunk_indices", []))
        if vanilla_set or dos_set:
            selected_jaccards.append(len(vanilla_set & dos_set) / max(len(vanilla_set | dos_set), 1))

    phen_rows.append(
        {
            "task": task,
            "dos_better_examples": dos_better,
            "vanilla_better_examples": vanilla_better,
            "tied_examples": tied,
            "dos_context_ge_95pct_doc": f"{near_full_dos}/{len(shared_ids)}",
            "dos_contiguous_selection": f"{dos_contiguous}/{len(shared_ids)}",
            "avg_selected_chunk_jaccard": round(mean(selected_jaccards), 4),
        }
    )

display(as_table(phen_rows))


,task,dos_better_examples,vanilla_better_examples,tied_examples,dos_context_ge_95pct_doc,dos_contiguous_selection,avg_selected_chunk_jaccard
0,qmsum,27,22,1,33/50,25/50,0.8727
1,qasper,17,13,20,50/50,50/50,0.9553
2,narrative_qa,11,6,33,0/50,0/50,0.4850
3,quality,6,2,42,50/50,50/50,0.9853
4,contract_nli,2,5,43,50/50,50/50,0.9715


## Main Findings

### 1. This is often not a pure retrieval contest

On several subset tasks, DOS is effectively seeing almost the whole document:

- `qasper`: `50/50` examples with DOS context at least 95% of the document
- `quality`: `50/50`
- `contract_nli`: `50/50`
- `qmsum`: `33/50`
- `narrative_qa`: `0/50`

So on `qasper`, `quality`, and `contract_nli`, the benchmark is often closer to:

- same or almost same evidence
- but arranged differently for the reader

### 2. Evidence overlap is extremely high on some tasks

Average Jaccard overlap between selected chunk sets is very high for:

- `quality`: about `0.985`
- `contract_nli`: about `0.972`
- `qasper`: about `0.955`

But much lower for:

- `narrative_qa`: about `0.485`

Interpretation:

- on `quality`, `qasper`, and `contract_nli`, DOS vs vanilla is largely an **ordering / source-fidelity test**
- on `narrative_qa`, DOS is still a more genuinely different retrieval behavior under real selection pressure

### 3. The task pattern makes sense

- `quality` benefits from coherent, contiguous reading order; DOS is strongest here
- `contract_nli` often wants clause-local evidence, and DOS can reintroduce too much early boilerplate before the decisive clause
- `qmsum` and `qasper` sit in the middle: DOS helps overall, but not uniformly example by example

### 4. The strongest current hypothesis

The strongest hypothesis supported by the saved outputs is:

> DOS helps mainly when preserving local discourse continuity or source structure helps the reader interpret the evidence, but hurts when the restored order buries the key local clause inside long, mostly irrelevant prefix text.


In [8]:
def inspect_example(task, example_id, preview_chars=260):
    rows = []
    for method in METHODS:
        by_id = {row["id"]: row for row in load_jsonl(RUN_ROOT / method / task / "results.jsonl")}
        row = by_id[example_id]
        rows.append(
            {
                "method": method,
                "score": round(score_row(task, row), 4),
                "prompt_ordering": row.get("prompt_ordering"),
                "num_context_chunks": row.get("num_context_chunks"),
                "context_over_document": round(row.get("context_tokens", 0) / max(row.get("document_tokens", 1), 1), 3),
                "selected_chunk_indices_head": row.get("selected_chunk_indices", [])[:20],
                "prediction_preview": row.get("prediction", "")[:preview_chars],
            }
        )
    return as_table(rows)


display(inspect_example("quality", "52845_75VB1ISR_1"))
display(inspect_example("contract_nli", "7_nda-11"))
display(inspect_example("qasper", "b6f15fb6279b82e34a5bf4828b7b5ddabfdf1d54"))


,method,score,prompt_ordering,num_context_chunks,context_over_document,selected_chunk_indices_head,prediction_preview
0,vanilla_rag,0.0,retrieval_rank,76,0.972,"[22, 42, 10, 64, 8, 61, 73, 38, 6, 41, 16, 33,...",12 years
1,dos_rag,100.0,document_order,76,0.991,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...",10 hours


,method,score,prompt_ordering,num_context_chunks,context_over_document,selected_chunk_indices_head,prediction_preview
0,vanilla_rag,100.0,retrieval_rank,78,1.009,"[19, 26, 8, 14, 25, 34, 38, 2, 40, 21, 15, 23,...",Not mentioned
1,dos_rag,100.0,document_order,76,1.019,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...",Not mentioned


,method,score,prompt_ordering,num_context_chunks,context_over_document,selected_chunk_indices_head,prediction_preview
0,vanilla_rag,100.0,retrieval_rank,68,0.999,"[48, 52, 10, 40, 60, 22, 1, 58, 21, 50, 38, 49...",multilingual NMT (MNMT) BIBREF19
1,dos_rag,100.0,document_order,68,1.010,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...",multilingual NMT (MNMT) BIBREF19


## What Is Still Unclear

This is the most important part for deciding what to do next.

### Unclear point 1: how much is ordering vs retrieval-core?

Right now, DOS differs from vanilla in two ways:

- it uses the official DOS retrieval core
- it restores the selected chunks to document order

Because both changed at once, the current benchmark does **not** isolate ordering cleanly.

### Unclear point 2: why exactly does `contract_nli` regress?

There are at least three plausible explanations:

- document-order restoration brings too much irrelevant boilerplate before the decisive clause
- chunk boundaries are not clause-aware enough for legal reasoning
- the reader is sensitive to local legal scope markers like exceptions, negations, and survival clauses

### Unclear point 3: when does near-full-document context stop helping?

The current subset used a 10k-token budget. That is large enough that several tasks nearly reconstruct the document. We still do not know whether DOS's gains remain when the budget is tighter and retrieval becomes harder.

### Unclear point 4: is the best direction generic ordering or task-adaptive ordering?

The current evidence points toward a task-adaptive story:

- narrative / reading-comprehension style tasks like structure preservation
- clause-local legal tasks may need a different ordering policy or chunk unit


## What To Do Next

### Highest-priority experiment: `reorder_only_rag`

Implement one new baseline:

1. use the exact same retrieved chunk set as `vanilla_rag`
2. only reorder those selected chunks into document order
3. keep everything else fixed

Why this is the best next step:

- it directly tests the main hypothesis from your current findings
- it is lightweight enough to finish soon
- it gives you a cleaner research claim than the current two-way comparison

### Second experiment: budget sweep on `quality` and `contract_nli`

Run `vanilla_rag`, `reorder_only_rag`, and `dos_rag` on a small budget sweep such as:

- `1500`
- `5000`
- `10000`

What this answers:

- whether DOS's gain is mainly a high-budget ordering effect
- whether the `contract_nli` loss worsens as more boilerplate is restored

### Third experiment: legal-focused improvement

If `contract_nli` stays weak, test a legal-specific variant:

- clause-aware chunking
- keep top relevant chunks near the front, then append document neighbors locally
- or reorder within a local window instead of reconstructing the whole prefix of the contract

### Best concrete improvement idea right now

The most promising concrete improvement is:

> **task-adaptive structure-preserving RAG**: preserve document order for narrative / long-form reasoning tasks, but use clause-aware or locally expanded retrieval for legal NLI.

This is stronger than jumping immediately to a full tree or graph system, because it is directly motivated by the effect you already observed in your own results.


## Reading Queue

Read these next, in this order.

### 1. DOS-RAG paper

- Laitenberger, Manning, and Liu. 2025. *Stronger Baselines for Retrieval-Augmented Generation with Long-Context Language Models*.
- Link: https://aclanthology.org/2025.emnlp-main.1656/
- Why: this is the paper your repo is explicitly trying to evaluate faithfully. Its main claim is that preserving source fidelity and document structure can make simple DOS-style retrieval surprisingly strong.

### 2. Lost in the Middle

- Liu et al. 2024. *Lost in the Middle: How Language Models Use Long Contexts*.
- Link: https://www-cs.stanford.edu/~nfliu/papers/lost-in-the-middle.arxiv2023.pdf
- Why: this is the key paper for understanding why prompt order can matter so much once long contexts get crowded.

### 3. OP-RAG / order-preserving RAG argument

- Yu, Xu, and Akkiraju. 2024. *In Defense of RAG in the Era of Long-Context Language Models*.
- Link: https://arxiv.org/abs/2409.01666
- Why: directly relevant because it argues for order-preserving retrieval and observes a sweet spot rather than monotonic gains from adding more retrieved text.

### 4. LongRAG

- Jiang, Ma, and Chen. 2024. *LongRAG: Enhancing Retrieval-Augmented Generation with Long-context LLMs*.
- Link: https://arxiv.org/abs/2406.15319
- Why: useful if your story becomes "larger retrieval units and more context-preserving units matter more than complicated pipelines."

### 5. Late Chunking

- Gunther et al. 2024/2025. *Late Chunking: Contextual Chunk Embeddings Using Long-Context Embedding Models*.
- Link: https://arxiv.org/abs/2409.04701
- Why: this is a very practical next improvement if you think chunk embeddings are losing too much cross-chunk context before retrieval.

### 6. TreeRAG

- Tao et al. 2025. *TreeRAG: Unleashing the Power of Hierarchical Storage for Enhanced Knowledge Retrieval in Long Documents*.
- Link: https://aclanthology.org/2025.findings-acl.20/
- Why: read this if you want to revisit the earlier tree-structured direction, but now with a clearer baseline story first.

### 7. SAKI-RAG

- Tao et al. 2025. *SAKI-RAG: Mitigating Context Fragmentation in Long-Document RAG via Sentence-level Attention Knowledge Integration*.
- Link: https://aclanthology.org/2025.emnlp-main.63/
- Why: highly relevant if your next hypothesis is that the problem is not just ordering, but broken inter-sentence continuity under standard chunking.

### 8. ContractNLI and ACORD

- Koreeda and Manning. 2021. *ContractNLI: A Dataset for Document-level Natural Language Inference for Contracts*.
- Link: https://arxiv.org/abs/2110.01799
- Why: explains why contract reasoning is hard, especially due to evidence localization, segmentation, and legal language phenomena like exceptions and negations.

- Wang et al. 2025. *ACORD: An Expert-Annotated Retrieval Dataset for Legal Contract Drafting*.
- Link: https://aclanthology.org/2025.acl-long.1206/
- Why: strong follow-up if you want to make the legal retrieval side of your story more serious and retrieval-centric.


## My Recommendation

If the goal is strong output in the next three weeks, the best direction is:

1. **finish the clean baseline story first**
2. **prove or disprove the ordering hypothesis directly**
3. **only then add one focused improvement**

Concretely, I would do:

- implement `reorder_only_rag`
- run it on `quality` and `contract_nli` first
- run a small context-budget sweep
- if the legal regression remains, implement a clause-aware or local-neighborhood ordering variant

That path gives you the cleanest possible research narrative:

- **Observation:** DOS beats vanilla overall, especially on `quality`, but loses on `contract_nli`.
- **Mechanism hypothesis:** the main effect is source-order preservation under long contexts.
- **Test:** reorder-only baseline and budget sweep.
- **Improvement:** task-adaptive structure-preserving retrieval.

That is a much tighter story than jumping straight into tree or graph retrieval without first resolving the signal that your current benchmark already surfaced.
